<a href="https://colab.research.google.com/github/loukaBl/Carbon_Footprint_Building_Analysis/blob/main/carbon_footprint_building_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Analyse de l'Empreinte Carbone des Bâtiments


## 1. Installation des Dépendances

In [25]:
!pip install pandas numpy matplotlib seaborn plotly ipywidgets scikit-learn requests openpyxl xlrd -q
print(" Toutes les dépendances sont installées!")

 Toutes les dépendances sont installées!


## 2. Importation des Bibliothèques

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
import requests
import io
import json

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Style pour les graphiques
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print(" Bibliothèques importées avec succès!")

 Bibliothèques importées avec succès!


## 3. Téléchargement et Préparation des Datasets Open Source

Nous utilisons plusieurs sources de données ouvertes:
- **ADEME** - Facteurs d'émission CO2 France
- **EPA** - Energy Star Building Data
- **UK Government** - Building Energy Efficiency
- **Données synthétiques basées sur des études réelles**

In [27]:
# DATASET 1: Facteurs d'émission CO2 par source d'énergie (ADEME/EPA standards)

emission_factors_data = {
    'source_energie': ['Electricite_France', 'Electricite_EU', 'Electricite_US',
                       'Gaz_Naturel', 'Fioul', 'Charbon', 'Biomasse', 'Solaire', 'Eolien'],
    'facteur_emission_kgCO2_kWh': [0.0569, 0.276, 0.417, 0.227, 0.324, 0.986, 0.015, 0.0, 0.0],
    'incertitude_percent': [5, 10, 8, 3, 5, 5, 20, 0, 0],
    'source': ['ADEME 2024', 'EEA 2023', 'EPA 2023', 'ADEME 2024', 'ADEME 2024',
               'IPCC 2023', 'ADEME 2024', 'ADEME 2024', 'ADEME 2024']
}

df_emission_factors = pd.DataFrame(emission_factors_data)
print("📊 Dataset 1: Facteurs d'émission CO2")
display(df_emission_factors)

📊 Dataset 1: Facteurs d'émission CO2


,source_energie,facteur_emission_kgCO2_kWh,incertitude_percent,source
0,Electricite_France,0.0569,5,ADEME 2024
1,Electricite_EU,0.2760,10,EEA 2023
2,Electricite_US,0.4170,8,EPA 2023
3,Gaz_Naturel,0.2270,3,ADEME 2024
4,Fioul,0.3240,5,ADEME 2024
5,Charbon,0.9860,5,IPCC 2023
6,Biomasse,0.0150,20,ADEME 2024
7,Solaire,0.0000,0,ADEME 2024
8,Eolien,0.0000,0,ADEME 2024


In [28]:
# DATASET 2: Consommation moyenne par type de bâtiment (données réelles agrégées)

np.random.seed(42)

building_types = ['Résidence', 'Maison Particulière', 'Appartement', 'Bureau',
                  'Commerce', 'Industrie', 'Entrepôt', 'École', 'Hôpital', 'Hôtel']

# Données basées sur les statistiques ADEME et Energy Star
building_stats = {
    'type_batiment': building_types * 50,
    'superficie_m2': np.concatenate([
        np.random.normal(150, 50, 50),   # Résidence
        np.random.normal(120, 40, 50),   # Maison
        np.random.normal(70, 25, 50),    # Appartement
        np.random.normal(500, 200, 50),  # Bureau
        np.random.normal(300, 150, 50),  # Commerce
        np.random.normal(2000, 800, 50), # Industrie
        np.random.normal(1500, 500, 50), # Entrepôt
        np.random.normal(3000, 1000, 50),# École
        np.random.normal(5000, 2000, 50),# Hôpital
        np.random.normal(2000, 800, 50)  # Hôtel
    ]),
    'nb_etages': np.concatenate([
        np.random.randint(1, 3, 50),
        np.random.randint(1, 3, 50),
        np.random.randint(1, 2, 50),
        np.random.randint(2, 10, 50),
        np.random.randint(1, 4, 50),
        np.random.randint(1, 5, 50),
        np.random.randint(1, 3, 50),
        np.random.randint(2, 5, 50),
        np.random.randint(3, 15, 50),
        np.random.randint(3, 20, 50)
    ]),
    'nb_pieces': np.concatenate([
        np.random.randint(4, 10, 50),
        np.random.randint(3, 8, 50),
        np.random.randint(2, 5, 50),
        np.random.randint(10, 50, 50),
        np.random.randint(2, 10, 50),
        np.random.randint(5, 30, 50),
        np.random.randint(2, 10, 50),
        np.random.randint(15, 50, 50),
        np.random.randint(50, 200, 50),
        np.random.randint(30, 150, 50)
    ]),
    'nb_occupants': np.concatenate([
        np.random.randint(2, 6, 50),
        np.random.randint(2, 5, 50),
        np.random.randint(1, 4, 50),
        np.random.randint(10, 100, 50),
        np.random.randint(5, 30, 50),
        np.random.randint(20, 200, 50),
        np.random.randint(5, 30, 50),
        np.random.randint(100, 500, 50),
        np.random.randint(200, 1000, 50),
        np.random.randint(50, 300, 50)
    ])
}

df_buildings_base = pd.DataFrame(building_stats)

# Calcul des consommations basé sur des formules réalistes (kWh/m²/an standards)
consumption_per_m2 = {
    'Résidence': {'elec': 110, 'gaz': 90, 'eau': 0.12},
    'Maison Particulière': {'elec': 100, 'gaz': 100, 'eau': 0.15},
    'Appartement': {'elec': 90, 'gaz': 80, 'eau': 0.10},
    'Bureau': {'elec': 150, 'gaz': 50, 'eau': 0.05},
    'Commerce': {'elec': 200, 'gaz': 60, 'eau': 0.08},
    'Industrie': {'elec': 300, 'gaz': 150, 'eau': 0.20},
    'Entrepôt': {'elec': 80, 'gaz': 40, 'eau': 0.03},
    'École': {'elec': 100, 'gaz': 70, 'eau': 0.10},
    'Hôpital': {'elec': 250, 'gaz': 120, 'eau': 0.25},
    'Hôtel': {'elec': 180, 'gaz': 90, 'eau': 0.18}
}

# Génération des consommations avec variation réaliste
df_buildings_base['conso_elec_kwh'] = df_buildings_base.apply(
    lambda x: max(0, x['superficie_m2'] * consumption_per_m2[x['type_batiment']]['elec'] *
                  np.random.uniform(0.7, 1.3)), axis=1
)
df_buildings_base['conso_gaz_kwh'] = df_buildings_base.apply(
    lambda x: max(0, x['superficie_m2'] * consumption_per_m2[x['type_batiment']]['gaz'] *
                  np.random.uniform(0.6, 1.4)), axis=1
)
df_buildings_base['conso_eau_m3'] = df_buildings_base.apply(
    lambda x: max(0, x['nb_occupants'] * 50 * np.random.uniform(0.8, 1.2)), axis=1  # ~50m³/personne/an
)

# Calcul de l'empreinte carbone (facteurs ADEME)
FACTEUR_ELEC = 0.0569  # kgCO2/kWh France
FACTEUR_GAZ = 0.227    # kgCO2/kWh
FACTEUR_EAU = 0.132    # kgCO2/m³ (traitement + distribution)

df_buildings_base['empreinte_carbone_kg'] = (
    df_buildings_base['conso_elec_kwh'] * FACTEUR_ELEC +
    df_buildings_base['conso_gaz_kwh'] * FACTEUR_GAZ +
    df_buildings_base['conso_eau_m3'] * FACTEUR_EAU
)

df_buildings_base['empreinte_carbone_tonnes'] = df_buildings_base['empreinte_carbone_kg'] / 1000

print(f" Dataset 2: Données de {len(df_buildings_base)} bâtiments générées")
display(df_buildings_base.head(10))

 Dataset 2: Données de 500 bâtiments générées


,type_batiment,superficie_m2,nb_etages,nb_pieces,nb_occupants,conso_elec_kwh,conso_gaz_kwh,conso_eau_m3,empreinte_carbone_kg,empreinte_carbone_tonnes
0,Résidence,174.835708,2,4,4,16426.572462,16396.264049,175.027428,4679.727533,4.679728
1,Maison Particulière,143.086785,2,4,5,15442.058782,9662.958626,271.979145,3108.046000,3.108046
2,Appartement,182.384427,2,9,2,12065.924305,12765.896615,114.677339,3599.547033,3.599547
3,Bureau,226.151493,1,9,2,26631.530913,9339.431559,98.157874,3648.341912,3.648342
4,Commerce,138.292331,2,6,5,23916.943787,7572.191281,282.143199,3117.004425,3.117004
5,Industrie,138.293152,1,9,2,43903.711974,25695.047809,102.186294,8344.385655,8.344386
6,Entrepôt,228.960641,1,7,3,20249.366021,8220.109522,158.225450,3039.039547,3.039040
7,École,188.371736,2,5,3,18189.743380,12676.480126,151.576708,3932.565512,3.932566
8,Hôpital,126.526281,1,5,5,30602.739082,13936.694389,246.317272,4937.439360,4.937439
9,Hôtel,177.128002,1,8,3,36178.008297,10756.164175,134.079207,4517.876395,4.517876


In [29]:
# DATASET 3: Actions de réduction et leur efficacité (données ADEME/études)
actions_reduction_data = {
    'action_id': list(range(1, 21)),
    'categorie': ['Isolation', 'Isolation', 'Isolation', 'Chauffage', 'Chauffage',
                  'Chauffage', 'Electricité', 'Electricité', 'Electricité', 'Electricité',
                  'Eau', 'Eau', 'Eau', 'Energie_Renouvelable', 'Energie_Renouvelable',
                  'Comportement', 'Comportement', 'Comportement', 'Equipement', 'Equipement'],
    'action': [
        'Isolation des combles', 'Isolation des murs extérieurs', 'Remplacement fenêtres double vitrage',
        'Installation pompe à chaleur', 'Remplacement chaudière haute performance', 'Installation thermostat intelligent',
        'Remplacement éclairage LED', 'Installation détecteurs de présence', 'Optimisation appareils électroménagers', 'Installation variateurs de vitesse',
        'Installation réducteurs de débit', 'Récupération eau de pluie', 'Remplacement chasse d\'eau économique',
        'Installation panneaux solaires photovoltaïques', 'Installation chauffe-eau solaire',
        'Sensibilisation occupants', 'Extinction appareils en veille', 'Optimisation température consigne',
        'Remplacement équipements classe A+++', 'Installation système de gestion énergétique (BMS)'
    ],
    'reduction_elec_percent': [5, 8, 10, 15, 5, 8, 25, 15, 20, 10, 0, 0, 0, 30, 5, 10, 8, 5, 15, 20],
    'reduction_gaz_percent': [25, 30, 15, 60, 25, 15, 0, 0, 0, 0, 0, 0, 0, 0, 40, 10, 0, 12, 0, 15],
    'reduction_eau_percent': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 30, 20, 25, 0, 0, 15, 0, 0, 10, 5],
    'cout_estime_euro': [3000, 12000, 8000, 15000, 5000, 300, 500, 800, 2000, 3000,
                         100, 2000, 200, 10000, 4000, 0, 0, 0, 3000, 8000],
    'difficulte': ['Moyenne', 'Élevée', 'Moyenne', 'Élevée', 'Moyenne', 'Faible',
                   'Faible', 'Faible', 'Moyenne', 'Moyenne', 'Faible', 'Moyenne', 'Faible',
                   'Élevée', 'Moyenne', 'Faible', 'Faible', 'Faible', 'Moyenne', 'Élevée'],
    'applicable_residence': [True, True, True, True, True, True, True, True, True, False,
                              True, True, True, True, True, True, True, True, True, False],
    'applicable_bureau': [True, True, True, True, True, True, True, True, True, True,
                          True, True, True, True, True, True, True, True, True, True],
    'applicable_industrie': [True, True, True, True, True, True, True, True, True, True,
                             True, True, True, True, True, True, True, True, True, True],
    'applicable_commerce': [True, True, True, True, True, True, True, True, True, True,
                            True, True, True, True, True, True, True, True, True, True],
    'temps_retour_annees': [3, 8, 10, 7, 5, 1, 1, 2, 4, 5, 0.5, 5, 0.5, 10, 6, 0, 0, 0, 5, 6]
}

df_actions = pd.DataFrame(actions_reduction_data)
print("📊 Dataset 3: Actions de réduction carbone")
display(df_actions)

📊 Dataset 3: Actions de réduction carbone


,action_id,categorie,action,reduction_elec_percent,reduction_gaz_percent,reduction_eau_percent,cout_estime_euro,difficulte,applicable_residence,applicable_bureau,applicable_industrie,applicable_commerce,temps_retour_annees
0,1,Isolation,Isolation des combles,5,25,0,3000,Moyenne,True,True,True,True,3.0
1,2,Isolation,Isolation des murs extérieurs,8,30,0,12000,Élevée,True,True,True,True,8.0
2,3,Isolation,Remplacement fenêtres double vitrage,10,15,0,8000,Moyenne,True,True,True,True,10.0
3,4,Chauffage,Installation pompe à chaleur,15,60,0,15000,Élevée,True,True,True,True,7.0
4,5,Chauffage,Remplacement chaudière haute performance,5,25,0,5000,Moyenne,True,True,True,True,5.0
5,6,Chauffage,Installation thermostat intelligent,8,15,0,300,Faible,True,True,True,True,1.0
6,7,Electricité,Remplacement éclairage LED,25,0,0,500,Faible,True,True,True,True,1.0
7,8,Electricité,Installation détecteurs de présence,15,0,0,800,Faible,True,True,True,True,2.0
8,9,Electricité,Optimisation appareils électroménagers,20,0,0,2000,Moyenne,True,True,True,True,4.0
9,10,Electricité,Installation variateurs de vitesse,10,0,0,3000,Moyenne,False,True,True,True,5.0


In [30]:
# DATASET 4: Benchmarks par type de bâtiment (normes RT2012/RE2020/Energy Star)

benchmarks_data = {
    'type_batiment': building_types,
    'conso_elec_kWh_m2_excellent': [50, 45, 40, 80, 100, 150, 40, 60, 150, 100],
    'conso_elec_kWh_m2_moyen': [110, 100, 90, 150, 200, 300, 80, 100, 250, 180],
    'conso_elec_kWh_m2_mauvais': [180, 170, 150, 250, 350, 500, 150, 170, 400, 300],
    'conso_gaz_kWh_m2_excellent': [40, 50, 35, 25, 30, 80, 20, 35, 60, 45],
    'conso_gaz_kWh_m2_moyen': [90, 100, 80, 50, 60, 150, 40, 70, 120, 90],
    'conso_gaz_kWh_m2_mauvais': [150, 170, 140, 90, 100, 250, 70, 120, 200, 150],
    'empreinte_kgCO2_m2_excellent': [5, 6, 4, 8, 10, 25, 4, 7, 20, 12],
    'empreinte_kgCO2_m2_moyen': [15, 16, 12, 18, 25, 55, 10, 15, 45, 30],
    'empreinte_kgCO2_m2_mauvais': [30, 32, 25, 35, 50, 100, 20, 30, 80, 55],
    'classe_DPE_seuil_A': ['< 70', '< 70', '< 65', '< 100', '< 120', '< 200', '< 60', '< 80', '< 180', '< 130'],
    'classe_DPE_seuil_G': ['> 450', '> 450', '> 400', '> 500', '> 600', '> 800', '> 350', '> 450', '> 700', '> 550']
}

df_benchmarks = pd.DataFrame(benchmarks_data)
print("Dataset 4: Benchmarks de performance énergétique")
display(df_benchmarks)

Dataset 4: Benchmarks de performance énergétique


,type_batiment,conso_elec_kWh_m2_excellent,conso_elec_kWh_m2_moyen,conso_elec_kWh_m2_mauvais,conso_gaz_kWh_m2_excellent,conso_gaz_kWh_m2_moyen,conso_gaz_kWh_m2_mauvais,empreinte_kgCO2_m2_excellent,empreinte_kgCO2_m2_moyen,empreinte_kgCO2_m2_mauvais,classe_DPE_seuil_A,classe_DPE_seuil_G
0,Résidence,50,110,180,40,90,150,5,15,30,< 70,> 450
1,Maison Particulière,45,100,170,50,100,170,6,16,32,< 70,> 450
2,Appartement,40,90,150,35,80,140,4,12,25,< 65,> 400
3,Bureau,80,150,250,25,50,90,8,18,35,< 100,> 500
4,Commerce,100,200,350,30,60,100,10,25,50,< 120,> 600
5,Industrie,150,300,500,80,150,250,25,55,100,< 200,> 800
6,Entrepôt,40,80,150,20,40,70,4,10,20,< 60,> 350
7,École,60,100,170,35,70,120,7,15,30,< 80,> 450
8,Hôpital,150,250,400,60,120,200,20,45,80,< 180,> 700
9,Hôtel,100,180,300,45,90,150,12,30,55,< 130,> 550


## 4. Nettoyage et Traitement des Données

In [31]:
def clean_dataset(df, name):
    """
    Fonction de nettoyage complète des datasets
    """
    print(f"\n Nettoyage du dataset: {name}")
    print(f"   Taille initiale: {df.shape}")

    # 1. Vérification des valeurs manquantes
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(f"    Valeurs manquantes détectées:")
        for col in missing[missing > 0].index:
            print(f"      - {col}: {missing[col]} valeurs")
            # Remplacement intelligent
            if df[col].dtype in ['float64', 'int64']:
                df[col].fillna(df[col].median(), inplace=True)
            else:
                df[col].fillna(df[col].mode()[0], inplace=True)
        print("    Valeurs manquantes remplacées")
    else:
        print("    Aucune valeur manquante")

    # 2. Vérification des valeurs négatives pour les colonnes numériques pertinentes
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if 'conso' in col.lower() or 'superficie' in col.lower() or 'nb_' in col.lower():
            neg_count = (df[col] < 0).sum()
            if neg_count > 0:
                print(f"    {neg_count} valeurs négatives dans {col} - corrigées")
                df[col] = df[col].abs()

    # 3. Suppression des doublons
    duplicates = df.duplicated().sum()
    if duplicates > 0:
        df.drop_duplicates(inplace=True)
        print(f"    {duplicates} doublons supprimés")

    # 4. Vérification des outliers (méthode IQR)
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        outliers = ((df[col] < (Q1 - 3 * IQR)) | (df[col] > (Q3 + 3 * IQR))).sum()
        if outliers > 0:
            # Capping des outliers extrêmes
            df[col] = df[col].clip(lower=Q1 - 3*IQR, upper=Q3 + 3*IQR)
            print(f"    {outliers} outliers traités dans {col}")

    print(f"   Taille finale: {df.shape}")
    return df

# Application du nettoyage
df_emission_factors = clean_dataset(df_emission_factors, "Facteurs d'émission")
df_buildings_base = clean_dataset(df_buildings_base, "Données bâtiments")
df_actions = clean_dataset(df_actions, "Actions de réduction")
df_benchmarks = clean_dataset(df_benchmarks, "Benchmarks")

print("\n" + "="*60)
print(" TOUS LES DATASETS SONT NETTOYÉS ET PRÊTS À L'UTILISATION")
print("="*60)


 Nettoyage du dataset: Facteurs d'émission
   Taille initiale: (9, 4)
    Aucune valeur manquante
   Taille finale: (9, 4)

 Nettoyage du dataset: Données bâtiments
   Taille initiale: (500, 10)
    Aucune valeur manquante
    2 valeurs négatives dans superficie_m2 - corrigées
    3 outliers traités dans superficie_m2
    22 outliers traités dans nb_etages
    15 outliers traités dans nb_pieces
    14 outliers traités dans nb_occupants
    12 outliers traités dans conso_elec_kwh
    8 outliers traités dans conso_gaz_kwh
    15 outliers traités dans conso_eau_m3
    7 outliers traités dans empreinte_carbone_kg
    7 outliers traités dans empreinte_carbone_tonnes
   Taille finale: (500, 10)

 Nettoyage du dataset: Actions de réduction
   Taille initiale: (20, 13)
    Aucune valeur manquante
    1 outliers traités dans reduction_eau_percent
   Taille finale: (20, 13)

 Nettoyage du dataset: Benchmarks
   Taille initiale: (10, 12)
    Aucune valeur manquante
   Taille finale: (10, 12)

 T

In [32]:
# Statistiques descriptives des datasets nettoyés
print("\n Statistiques du dataset des bâtiments:")
display(df_buildings_base.describe().round(2))

print("\n Distribution par type de bâtiment:")
display(df_buildings_base['type_batiment'].value_counts())


 Statistiques du dataset des bâtiments:


,superficie_m2,nb_etages,nb_pieces,nb_occupants,conso_elec_kwh,conso_gaz_kwh,conso_eau_m3,empreinte_carbone_kg,empreinte_carbone_tonnes
count,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00
mean,1470.16,3.53,30.70,125.52,219788.53,119908.80,6149.38,40950.22,40.95
std,1724.82,3.26,39.61,186.79,277713.70,149544.11,9034.50,50531.02,50.53
min,3.77,1.00,2.00,1.00,0.00,0.00,40.83,123.70,0.12
25%,152.78,1.00,5.00,4.00,22717.63,12105.26,204.67,4103.99,4.10
50%,780.23,2.00,9.00,25.00,105046.67,57577.75,1251.87,19540.03,19.54
75%,2177.50,4.00,42.00,192.00,315895.24,169921.14,9135.36,60025.70,60.03
max,8251.66,13.00,153.00,756.00,1195428.07,643368.80,35927.41,227790.84,227.79



 Distribution par type de bâtiment:


,count
type_batiment,
Résidence,50
Maison Particulière,50
Appartement,50
Bureau,50
Commerce,50
Industrie,50
Entrepôt,50
École,50
Hôpital,50


## 5. Entraînement du Modèle de Prédiction

In [33]:
# Préparation des données pour le modèle
le = LabelEncoder()
df_model = df_buildings_base.copy()
df_model['type_batiment_encoded'] = le.fit_transform(df_model['type_batiment'])

# Features et target
features = ['superficie_m2', 'nb_etages', 'nb_pieces', 'nb_occupants',
            'conso_elec_kwh', 'conso_gaz_kwh', 'conso_eau_m3', 'type_batiment_encoded']
X = df_model[features]
y = df_model['empreinte_carbone_kg']

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Entraînement du modèle Random Forest
model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)
model.fit(X_train_scaled, y_train)

# Évaluation
train_score = model.score(X_train_scaled, y_train)
test_score = model.score(X_test_scaled, y_test)

print(f" Performance du modèle:")
print(f"   Score R² (train): {train_score:.4f}")
print(f"   Score R² (test):  {test_score:.4f}")

# Importance des features
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n Importance des variables:")
display(feature_importance)

 Performance du modèle:
   Score R² (train): 0.9992
   Score R² (test):  0.9874

 Importance des variables:


,feature,importance
5,conso_gaz_kwh,0.973084
4,conso_elec_kwh,0.019781
0,superficie_m2,0.004032
6,conso_eau_m3,0.000756
2,nb_pieces,0.000709
7,type_batiment_encoded,0.000557
3,nb_occupants,0.000555
1,nb_etages,0.000526


## 6. Fonctions de Calcul d'Empreinte Carbone

In [34]:
class CarbonFootprintCalculator:
    """
    Classe pour calculer l'empreinte carbone et recommander des actions
    """

    def __init__(self):
        # Facteurs d'émission (kgCO2/unité)
        self.facteur_elec = 0.0569  # kgCO2/kWh (France)
        self.facteur_gaz = 0.227    # kgCO2/kWh
        self.facteur_eau = 0.132    # kgCO2/m³

        # Mapping des types de bâtiments
        self.building_mapping = {
            'Résidence': 'residence',
            'Maison Particulière': 'residence',
            'Appartement': 'residence',
            'Bureau': 'bureau',
            'Commerce': 'commerce',
            'Industrie': 'industrie',
            'Entrepôt': 'industrie',
            'École': 'bureau',
            'Hôpital': 'industrie',
            'Hôtel': 'commerce'
        }

    def calculer_empreinte(self, conso_elec, conso_gaz, conso_eau):
        """
        Calcule l'empreinte carbone totale en kgCO2
        """
        empreinte_elec = conso_elec * self.facteur_elec
        empreinte_gaz = conso_gaz * self.facteur_gaz
        empreinte_eau = conso_eau * self.facteur_eau

        total = empreinte_elec + empreinte_gaz + empreinte_eau

        return {
            'electricite': round(empreinte_elec, 2),
            'gaz': round(empreinte_gaz, 2),
            'eau': round(empreinte_eau, 2),
            'total_kg': round(total, 2),
            'total_tonnes': round(total / 1000, 3)
        }

    def obtenir_classe_dpe(self, empreinte_kg, superficie):
        """
        Détermine la classe DPE approximative
        """
        empreinte_m2 = empreinte_kg / superficie if superficie > 0 else 0

        if empreinte_m2 < 6:
            return 'A', '#319834', 'Excellent'
        elif empreinte_m2 < 11:
            return 'B', '#33cc66', 'Très bon'
        elif empreinte_m2 < 20:
            return 'C', '#cbfc33', 'Bon'
        elif empreinte_m2 < 35:
            return 'D', '#fbea02', 'Moyen'
        elif empreinte_m2 < 55:
            return 'E', '#fccc04', 'Insuffisant'
        elif empreinte_m2 < 80:
            return 'F', '#fc9905', 'Mauvais'
        else:
            return 'G', '#fc0505', 'Très mauvais'

    def recommander_actions(self, type_batiment, empreinte_details, conso_elec, conso_gaz, conso_eau):
        """
        Recommande des actions personnalisées selon le type de bâtiment
        """
        actions_recommandees = []

        # Déterminer la catégorie de bâtiment
        categorie = self.building_mapping.get(type_batiment, 'residence')

        # Sélectionner les colonnes d'applicabilité
        if categorie == 'residence':
            applicable_col = 'applicable_residence'
        elif categorie == 'bureau':
            applicable_col = 'applicable_bureau'
        elif categorie == 'industrie':
            applicable_col = 'applicable_industrie'
        else:
            applicable_col = 'applicable_commerce'

        # Filtrer les actions applicables
        actions_applicables = df_actions[df_actions[applicable_col] == True].copy()

        # Calculer l'impact potentiel de chaque action
        for _, action in actions_applicables.iterrows():
            reduction_elec = conso_elec * (action['reduction_elec_percent'] / 100) * self.facteur_elec
            reduction_gaz = conso_gaz * (action['reduction_gaz_percent'] / 100) * self.facteur_gaz
            reduction_eau = conso_eau * (action['reduction_eau_percent'] / 100) * self.facteur_eau

            reduction_totale = reduction_elec + reduction_gaz + reduction_eau
            reduction_percent = (reduction_totale / empreinte_details['total_kg']) * 100 if empreinte_details['total_kg'] > 0 else 0

            if reduction_totale > 0:
                actions_recommandees.append({
                    'action_id': action['action_id'],
                    'categorie': action['categorie'],
                    'action': action['action'],
                    'reduction_kg': round(reduction_totale, 2),
                    'reduction_percent': round(reduction_percent, 2),
                    'cout_euro': action['cout_estime_euro'],
                    'difficulte': action['difficulte'],
                    'temps_retour': action['temps_retour_annees'],
                    'priorite': round(reduction_percent / max(action['cout_estime_euro'], 1) * 1000, 2)
                })

        # Trier par priorité (meilleur rapport coût/efficacité)
        actions_recommandees = sorted(actions_recommandees, key=lambda x: x['priorite'], reverse=True)

        return actions_recommandees

    def simuler_reduction(self, empreinte_initiale, actions_selectionnees, conso_elec, conso_gaz, conso_eau):
        """
        Simule la réduction d'empreinte après application des actions
        """
        reduction_elec_total = 0
        reduction_gaz_total = 0
        reduction_eau_total = 0
        cout_total = 0

        details_actions = []

        for action_id in actions_selectionnees:
            action = df_actions[df_actions['action_id'] == action_id].iloc[0]

            red_elec = action['reduction_elec_percent'] / 100
            red_gaz = action['reduction_gaz_percent'] / 100
            red_eau = action['reduction_eau_percent'] / 100

            # Éviter le cumul excessif (plafond à 80%)
            reduction_elec_total = min(0.8, reduction_elec_total + red_elec * (1 - reduction_elec_total))
            reduction_gaz_total = min(0.8, reduction_gaz_total + red_gaz * (1 - reduction_gaz_total))
            reduction_eau_total = min(0.8, reduction_eau_total + red_eau * (1 - reduction_eau_total))

            cout_total += action['cout_estime_euro']

            details_actions.append({
                'action': action['action'],
                'reduction_elec': f"{red_elec*100:.0f}%",
                'reduction_gaz': f"{red_gaz*100:.0f}%",
                'reduction_eau': f"{red_eau*100:.0f}%"
            })

        # Calcul nouvelle empreinte
        nouvelle_conso_elec = conso_elec * (1 - reduction_elec_total)
        nouvelle_conso_gaz = conso_gaz * (1 - reduction_gaz_total)
        nouvelle_conso_eau = conso_eau * (1 - reduction_eau_total)

        nouvelle_empreinte = self.calculer_empreinte(nouvelle_conso_elec, nouvelle_conso_gaz, nouvelle_conso_eau)

        reduction_absolue = empreinte_initiale['total_kg'] - nouvelle_empreinte['total_kg']
        reduction_relative = (reduction_absolue / empreinte_initiale['total_kg']) * 100 if empreinte_initiale['total_kg'] > 0 else 0

        return {
            'empreinte_initiale': empreinte_initiale,
            'nouvelle_empreinte': nouvelle_empreinte,
            'reduction_absolue_kg': round(reduction_absolue, 2),
            'reduction_relative_percent': round(reduction_relative, 2),
            'cout_total': cout_total,
            'details_actions': details_actions,
            'nouvelles_conso': {
                'elec': round(nouvelle_conso_elec, 2),
                'gaz': round(nouvelle_conso_gaz, 2),
                'eau': round(nouvelle_conso_eau, 2)
            }
        }

# Instanciation du calculateur
calculator = CarbonFootprintCalculator()
print(" Calculateur d'empreinte carbone initialisé!")

 Calculateur d'empreinte carbone initialisé!


## 7. Dashboard Interactif

In [35]:
# Style CSS personnalisé pour le dashboard
dashboard_css = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');

.dashboard-container {
    font-family: 'Inter', sans-serif;
    max-width: 1200px;
    margin: 0 auto;
}

.dashboard-header {
    background: linear-gradient(135deg, #1a365d 0%, #2d5a87 100%);
    color: white;
    padding: 30px;
    border-radius: 16px;
    margin-bottom: 24px;
    text-align: center;
}

.dashboard-header h1 {
    margin: 0;
    font-size: 28px;
    font-weight: 700;
}

.dashboard-header p {
    margin: 10px 0 0;
    opacity: 0.9;
    font-size: 16px;
}

.metric-card {
    background: white;
    border-radius: 12px;
    padding: 20px;
    box-shadow: 0 4px 6px rgba(0, 0, 0, 0.07);
    border: 1px solid #e5e7eb;
    margin-bottom: 16px;
}

.metric-card h3 {
    color: #6b7280;
    font-size: 14px;
    font-weight: 500;
    margin: 0 0 8px;
    text-transform: uppercase;
    letter-spacing: 0.5px;
}

.metric-value {
    font-size: 32px;
    font-weight: 700;
    color: #1f2937;
}

.metric-unit {
    font-size: 16px;
    color: #6b7280;
    font-weight: 400;
}

.dpe-badge {
    display: inline-block;
    padding: 8px 24px;
    border-radius: 8px;
    font-size: 24px;
    font-weight: 700;
    color: white;
}

.action-card {
    background: white;
    border-radius: 12px;
    padding: 16px;
    margin-bottom: 12px;
    border-left: 4px solid #3b82f6;
    box-shadow: 0 2px 4px rgba(0, 0, 0, 0.05);
}

.action-card.selected {
    border-left-color: #10b981;
    background: #f0fdf4;
}

.action-title {
    font-weight: 600;
    color: #1f2937;
    margin-bottom: 8px;
}

.action-meta {
    display: flex;
    gap: 16px;
    font-size: 13px;
    color: #6b7280;
}

.reduction-badge {
    background: #dcfce7;
    color: #166534;
    padding: 4px 12px;
    border-radius: 20px;
    font-weight: 600;
    font-size: 13px;
}

.results-container {
    background: linear-gradient(135deg, #ecfdf5 0%, #d1fae5 100%);
    border-radius: 16px;
    padding: 24px;
    margin-top: 24px;
}

.comparison-arrow {
    font-size: 32px;
    color: #10b981;
}

.section-title {
    font-size: 20px;
    font-weight: 600;
    color: #1f2937;
    margin-bottom: 16px;
    padding-bottom: 8px;
    border-bottom: 2px solid #e5e7eb;
}
</style>
"""

display(HTML(dashboard_css))

In [36]:
# Création des widgets d'entrée
style = {'description_width': '180px'}
layout = widgets.Layout(width='400px')

# Inputs utilisateur
type_batiment_widget = widgets.Dropdown(
    options=['Résidence', 'Maison Particulière', 'Appartement', 'Bureau',
             'Commerce', 'Industrie', 'Entrepôt', 'École', 'Hôpital', 'Hôtel'],
    value='Maison Particulière',
    description='Type de bâtiment:',
    style=style,
    layout=layout
)

superficie_widget = widgets.FloatSlider(
    value=120,
    min=20,
    max=5000,
    step=10,
    description='Superficie (m²):',
    style=style,
    layout=layout
)

nb_etages_widget = widgets.IntSlider(
    value=2,
    min=1,
    max=20,
    description='Nombre d\'étages:',
    style=style,
    layout=layout
)

nb_pieces_widget = widgets.IntSlider(
    value=5,
    min=1,
    max=50,
    description='Nombre de pièces:',
    style=style,
    layout=layout
)

nb_occupants_widget = widgets.IntSlider(
    value=4,
    min=1,
    max=100,
    description='Nombre d\'occupants:',
    style=style,
    layout=layout
)

conso_elec_widget = widgets.FloatText(
    value=8500,
    description='Conso. électrique (kWh/an):',
    style=style,
    layout=layout
)

conso_gaz_widget = widgets.FloatText(
    value=12000,
    description='Conso. gaz (kWh/an):',
    style=style,
    layout=layout
)

conso_eau_widget = widgets.FloatText(
    value=150,
    description='Conso. eau (m³/an):',
    style=style,
    layout=layout
)

# Bouton de calcul
calculate_button = widgets.Button(
    description=' Calculer l\'empreinte carbone',
    button_style='primary',
    layout=widgets.Layout(width='300px', height='45px'),
    style={'font_weight': 'bold'}
)

# Zone de sortie
output = widgets.Output()

# Variable globale pour stocker les actions recommandées
current_recommendations = []
current_empreinte = {}
current_params = {}

In [37]:
def create_empreinte_pie_chart(empreinte):
    """
    Crée un graphique en camembert de la répartition de l'empreinte
    """
    labels = ['Électricité', 'Gaz', 'Eau']
    values = [empreinte['electricite'], empreinte['gaz'], empreinte['eau']]
    colors = ['#3b82f6', '#f59e0b', '#06b6d4']

    fig = go.Figure(data=[go.Pie(
        labels=labels,
        values=values,
        hole=0.5,
        marker_colors=colors,
        textinfo='label+percent',
        textfont_size=14,
        hovertemplate='<b>%{label}</b><br>%{value:.1f} kgCO2<br>%{percent}<extra></extra>'
    )])

    fig.update_layout(
        title=dict(text='Répartition de l\'empreinte carbone', font=dict(size=18)),
        showlegend=True,
        legend=dict(orientation='h', yanchor='bottom', y=-0.2),
        height=400,
        annotations=[dict(
            text=f'{empreinte["total_kg"]:.0f}<br>kgCO2',
            x=0.5, y=0.5, font_size=20, showarrow=False
        )]
    )

    return fig

def create_comparison_chart(type_batiment, empreinte_m2):
    """
    Compare l'empreinte avec les benchmarks
    """
    benchmark = df_benchmarks[df_benchmarks['type_batiment'] == type_batiment].iloc[0]

    categories = ['Excellent', 'Votre bâtiment', 'Moyen', 'Mauvais']
    values = [
        benchmark['empreinte_kgCO2_m2_excellent'],
        empreinte_m2,
        benchmark['empreinte_kgCO2_m2_moyen'],
        benchmark['empreinte_kgCO2_m2_mauvais']
    ]
    colors = ['#10b981', '#3b82f6', '#f59e0b', '#ef4444']

    fig = go.Figure(data=[
        go.Bar(
            x=categories,
            y=values,
            marker_color=colors,
            text=[f'{v:.1f}' for v in values],
            textposition='outside'
        )
    ])

    fig.update_layout(
        title=dict(text='Comparaison avec les benchmarks (kgCO2/m²/an)', font=dict(size=18)),
        yaxis_title='kgCO2/m²/an',
        height=400,
        showlegend=False
    )

    return fig

def create_actions_impact_chart(actions):
    """
    Crée un graphique des impacts des actions recommandées
    """
    if not actions:
        return None

    top_actions = actions[:10]  # Top 10 actions

    fig = go.Figure(data=[
        go.Bar(
            y=[a['action'][:40] + '...' if len(a['action']) > 40 else a['action'] for a in top_actions],
            x=[a['reduction_percent'] for a in top_actions],
            orientation='h',
            marker_color='#10b981',
            text=[f"{a['reduction_percent']:.1f}%" for a in top_actions],
            textposition='outside'
        )
    ])

    fig.update_layout(
        title=dict(text='Top 10 Actions de Réduction (% de réduction)', font=dict(size=18)),
        xaxis_title='% de réduction',
        height=450,
        margin=dict(l=250),
        showlegend=False
    )

    return fig

In [38]:
def on_calculate_click(b):
    """
    Fonction appelée lors du clic sur le bouton de calcul
    """
    global current_recommendations, current_empreinte, current_params

    with output:
        clear_output(wait=True)

        # Récupération des valeurs
        type_batiment = type_batiment_widget.value
        superficie = superficie_widget.value
        nb_etages = nb_etages_widget.value
        nb_pieces = nb_pieces_widget.value
        nb_occupants = nb_occupants_widget.value
        conso_elec = conso_elec_widget.value
        conso_gaz = conso_gaz_widget.value
        conso_eau = conso_eau_widget.value

        # Stockage des paramètres
        current_params = {
            'type_batiment': type_batiment,
            'superficie': superficie,
            'conso_elec': conso_elec,
            'conso_gaz': conso_gaz,
            'conso_eau': conso_eau
        }

        # Calcul de l'empreinte
        empreinte = calculator.calculer_empreinte(conso_elec, conso_gaz, conso_eau)
        current_empreinte = empreinte

        # Classe DPE
        classe, couleur, description = calculator.obtenir_classe_dpe(empreinte['total_kg'], superficie)
        empreinte_m2 = empreinte['total_kg'] / superficie

        # Recommandations
        recommendations = calculator.recommander_actions(
            type_batiment, empreinte, conso_elec, conso_gaz, conso_eau
        )
        current_recommendations = recommendations

        # Affichage du header
        display(HTML(f"""
        <div class="dashboard-container">
            <div class="dashboard-header">
                <h1> Résultats de l'Analyse</h1>
                <p>{type_batiment} • {superficie:.0f} m² • {nb_occupants} occupants</p>
            </div>
        </div>
        """))

        # Métriques principales
        display(HTML(f"""
        <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 16px; margin-bottom: 24px;">
            <div class="metric-card">
                <h3>Empreinte Totale</h3>
                <div class="metric-value">{empreinte['total_kg']:.0f} <span class="metric-unit">kgCO2/an</span></div>
            </div>
            <div class="metric-card">
                <h3>Empreinte par m²</h3>
                <div class="metric-value">{empreinte_m2:.1f} <span class="metric-unit">kgCO2/m²/an</span></div>
            </div>
            <div class="metric-card">
                <h3>Par Occupant</h3>
                <div class="metric-value">{empreinte['total_kg']/nb_occupants:.0f} <span class="metric-unit">kgCO2/pers</span></div>
            </div>
            <div class="metric-card">
                <h3>Classe Énergétique</h3>
                <div class="dpe-badge" style="background-color: {couleur};">{classe}</div>
                <div style="margin-top: 8px; color: #6b7280; font-size: 14px;">{description}</div>
            </div>
        </div>
        """))

        # Graphiques
        fig_pie = create_empreinte_pie_chart(empreinte)
        fig_comparison = create_comparison_chart(type_batiment, empreinte_m2)

        # Affichage côte à côte
        from plotly.subplots import make_subplots
        fig_combined = make_subplots(
            rows=1, cols=2,
            specs=[[{'type': 'pie'}, {'type': 'bar'}]],
            subplot_titles=('Répartition de l\'empreinte', 'Comparaison aux benchmarks')
        )

        fig_pie.show()
        fig_comparison.show()

        # Section Actions
        display(HTML('<h2 class="section-title"> Actions de Réduction Recommandées</h2>'))

        if recommendations:
            fig_actions = create_actions_impact_chart(recommendations)
            if fig_actions:
                fig_actions.show()

            # Liste détaillée des actions
            actions_html = '<div style="display: grid; grid-template-columns: repeat(2, 1fr); gap: 12px;">'
            for i, action in enumerate(recommendations[:10]):
                difficulty_color = {'Faible': '#10b981', 'Moyenne': '#f59e0b', 'Élevée': '#ef4444'}
                actions_html += f"""
                <div class="action-card">
                    <div class="action-title">
                        <span style="color: #6b7280;">#{i+1}</span> {action['action']}
                    </div>
                    <div class="action-meta">
                        <span class="reduction-badge">-{action['reduction_percent']:.1f}%</span>
                        <span> {action['cout_euro']:,.0f}€</span>
                        <span style="color: {difficulty_color.get(action['difficulte'], '#6b7280')}">⚡ {action['difficulte']}</span>
                        <span> ROI: {action['temps_retour']} ans</span>
                    </div>
                </div>
                """
            actions_html += '</div>'
            display(HTML(actions_html))
        else:
            display(HTML('<p>Aucune action de réduction applicable trouvée.</p>'))

# Connecter le bouton à la fonction
calculate_button.on_click(on_calculate_click)

In [39]:
# Affichage du formulaire d'entrée
display(HTML("""
<div class="dashboard-container">
    <div class="dashboard-header">
        <h1> Calculateur d'Empreinte Carbone</h1>
        <p>Entrez les informations de votre bâtiment pour calculer son empreinte carbone</p>
    </div>
</div>
"""))

display(HTML('<h3 class="section-title"> Informations du Bâtiment</h3>'))

# Organisation des widgets en grille
input_box = widgets.VBox([
    widgets.HBox([type_batiment_widget]),
    widgets.HBox([superficie_widget, nb_etages_widget]),
    widgets.HBox([nb_pieces_widget, nb_occupants_widget]),
])

display(input_box)

display(HTML('<h3 class="section-title"> Consommations Annuelles</h3>'))

conso_box = widgets.VBox([
    conso_elec_widget,
    conso_gaz_widget,
    conso_eau_widget,
])

display(conso_box)
display(widgets.HTML('<br>'))
display(calculate_button)
display(output)

HTML(value='<br>')

Button(button_style='primary', description=" Calculer l'empreinte carbone", layout=Layout(height='45px', width…

Output()

## 8. Simulateur d'Actions avec Sélection Interactive

In [40]:
# Créer les widgets pour la sélection des actions
actions_checkboxes = []
actions_output = widgets.Output()

def create_action_selector():
    """
    Crée le sélecteur d'actions interactif
    """
    global actions_checkboxes
    actions_checkboxes = []

    display(HTML("""
    <div class="dashboard-header" style="background: linear-gradient(135deg, #065f46 0%, #10b981 100%);">
        <h1> Simulateur de Plan d'Actions</h1>
        <p>Sélectionnez les actions que vous souhaitez mettre en place</p>
    </div>
    """))

    if not current_recommendations:
        display(HTML('<p style="color: #ef4444;"> Veuillez d\'abord calculer l\'empreinte carbone ci-dessus.</p>'))
        return

    # Créer les checkboxes pour chaque action
    for action in current_recommendations[:15]:
        cb = widgets.Checkbox(
            value=False,
            description=f"{action['action']} (-{action['reduction_percent']:.1f}%, {action['cout_euro']:,.0f}€)",
            indent=False,
            layout=widgets.Layout(width='100%')
        )
        cb.action_id = action['action_id']
        actions_checkboxes.append(cb)

    # Bouton de simulation
    simulate_button = widgets.Button(
        description=' Simuler la réduction',
        button_style='success',
        layout=widgets.Layout(width='250px', height='45px')
    )

    def on_simulate_click(b):
        with actions_output:
            clear_output(wait=True)

            selected_ids = [cb.action_id for cb in actions_checkboxes if cb.value]

            if not selected_ids:
                display(HTML('<p style="color: #f59e0b;"> Veuillez sélectionner au moins une action.</p>'))
                return

            # Simulation
            result = calculator.simuler_reduction(
                current_empreinte,
                selected_ids,
                current_params['conso_elec'],
                current_params['conso_gaz'],
                current_params['conso_eau']
            )

            # Affichage des résultats
            display(HTML(f"""
            <div class="results-container">
                <h2 style="color: #065f46; margin-top: 0;"> Résultats de la Simulation</h2>

                <div style="display: grid; grid-template-columns: 1fr auto 1fr; gap: 24px; align-items: center; margin: 24px 0;">
                    <div class="metric-card" style="text-align: center;">
                        <h3>Empreinte Actuelle</h3>
                        <div class="metric-value" style="color: #ef4444;">
                            {result['empreinte_initiale']['total_kg']:.0f}
                            <span class="metric-unit">kgCO2/an</span>
                        </div>
                    </div>

                    <div style="text-align: center;">
                        <div class="comparison-arrow">→</div>
                        <div class="reduction-badge" style="font-size: 18px; padding: 8px 16px;">
                            -{result['reduction_relative_percent']:.1f}%
                        </div>
                    </div>

                    <div class="metric-card" style="text-align: center;">
                        <h3>Nouvelle Empreinte</h3>
                        <div class="metric-value" style="color: #10b981;">
                            {result['nouvelle_empreinte']['total_kg']:.0f}
                            <span class="metric-unit">kgCO2/an</span>
                        </div>
                    </div>
                </div>

                <div style="display: grid; grid-template-columns: repeat(3, 1fr); gap: 16px;">
                    <div class="metric-card">
                        <h3>Réduction Absolue</h3>
                        <div class="metric-value" style="color: #10b981;">
                            -{result['reduction_absolue_kg']:.0f}
                            <span class="metric-unit">kgCO2</span>
                        </div>
                    </div>
                    <div class="metric-card">
                        <h3>Investissement Total</h3>
                        <div class="metric-value">
                            {result['cout_total']:,.0f}
                            <span class="metric-unit">€</span>
                        </div>
                    </div>
                    <div class="metric-card">
                        <h3>Économie CO2/€</h3>
                        <div class="metric-value" style="color: #3b82f6;">
                            {result['reduction_absolue_kg']/max(result['cout_total'],1)*100:.1f}
                            <span class="metric-unit">kgCO2/100€</span>
                        </div>
                    </div>
                </div>
            </div>
            """))

            # Graphique avant/après
            fig = go.Figure()

            categories = ['Électricité', 'Gaz', 'Eau']

            fig.add_trace(go.Bar(
                name='Avant',
                x=categories,
                y=[result['empreinte_initiale']['electricite'],
                   result['empreinte_initiale']['gaz'],
                   result['empreinte_initiale']['eau']],
                marker_color='#ef4444'
            ))

            fig.add_trace(go.Bar(
                name='Après',
                x=categories,
                y=[result['nouvelle_empreinte']['electricite'],
                   result['nouvelle_empreinte']['gaz'],
                   result['nouvelle_empreinte']['eau']],
                marker_color='#10b981'
            ))

            fig.update_layout(
                title='Comparaison Avant/Après par Source',
                yaxis_title='kgCO2/an',
                barmode='group',
                height=400
            )

            fig.show()

            # Tableau détaillé des actions
            display(HTML('<h3 class="section-title"> Détail des Actions Sélectionnées</h3>'))

            actions_detail_html = '<table style="width: 100%; border-collapse: collapse;">'
            actions_detail_html += '''
                <tr style="background: #f3f4f6;">
                    <th style="padding: 12px; text-align: left;">Action</th>
                    <th style="padding: 12px; text-align: center;">Réd. Élec.</th>
                    <th style="padding: 12px; text-align: center;">Réd. Gaz</th>
                    <th style="padding: 12px; text-align: center;">Réd. Eau</th>
                </tr>
            '''

            for detail in result['details_actions']:
                actions_detail_html += f'''
                <tr style="border-bottom: 1px solid #e5e7eb;">
                    <td style="padding: 12px;">{detail['action']}</td>
                    <td style="padding: 12px; text-align: center; color: #3b82f6;">{detail['reduction_elec']}</td>
                    <td style="padding: 12px; text-align: center; color: #f59e0b;">{detail['reduction_gaz']}</td>
                    <td style="padding: 12px; text-align: center; color: #06b6d4;">{detail['reduction_eau']}</td>
                </tr>
                '''

            actions_detail_html += '</table>'
            display(HTML(actions_detail_html))

    simulate_button.on_click(on_simulate_click)

    # Affichage
    display(HTML('<h3 class="section-title"> Sélectionnez vos actions:</h3>'))
    for cb in actions_checkboxes:
        display(cb)
    display(widgets.HTML('<br>'))
    display(simulate_button)
    display(actions_output)

# Exécuter le sélecteur
create_action_selector()

## 9. Analyse Exploratoire des Datasets

In [41]:
# Visualisation de la distribution des émissions par type de bâtiment
fig = px.box(
    df_buildings_base,
    x='type_batiment',
    y='empreinte_carbone_tonnes',
    color='type_batiment',
    title='Distribution de l\'Empreinte Carbone par Type de Bâtiment'
)

fig.update_layout(
    xaxis_title='Type de Bâtiment',
    yaxis_title='Empreinte Carbone (tonnes CO2/an)',
    showlegend=False,
    xaxis_tickangle=-45,
    height=500
)

fig.show()

In [42]:
# Corrélation entre les variables
numeric_cols = ['superficie_m2', 'nb_etages', 'nb_pieces', 'nb_occupants',
                'conso_elec_kwh', 'conso_gaz_kwh', 'conso_eau_m3', 'empreinte_carbone_kg']

correlation_matrix = df_buildings_base[numeric_cols].corr()

fig = px.imshow(
    correlation_matrix,
    labels=dict(color="Corrélation"),
    x=numeric_cols,
    y=numeric_cols,
    color_continuous_scale='RdBu_r',
    aspect='auto',
    title='Matrice de Corrélation des Variables'
)

fig.update_layout(height=600)
fig.show()

In [43]:
# Scatter plot: Superficie vs Empreinte Carbone
fig = px.scatter(
    df_buildings_base,
    x='superficie_m2',
    y='empreinte_carbone_tonnes',
    color='type_batiment',
    size='nb_occupants',
    hover_data=['conso_elec_kwh', 'conso_gaz_kwh'],
    title='Relation Superficie - Empreinte Carbone',
    trendline='ols'
)

fig.update_layout(
    xaxis_title='Superficie (m²)',
    yaxis_title='Empreinte Carbone (tonnes CO2/an)',
    height=600
)

fig.show()

In [44]:
# Statistiques moyennes par type de bâtiment
stats_by_type = df_buildings_base.groupby('type_batiment').agg({
    'superficie_m2': 'mean',
    'nb_occupants': 'mean',
    'conso_elec_kwh': 'mean',
    'conso_gaz_kwh': 'mean',
    'conso_eau_m3': 'mean',
    'empreinte_carbone_tonnes': 'mean'
}).round(2)

stats_by_type.columns = ['Superficie Moy. (m²)', 'Occupants Moy.', 'Conso Élec. (kWh)',
                          'Conso Gaz (kWh)', 'Conso Eau (m³)', 'Empreinte (tCO2)']

print(" Statistiques Moyennes par Type de Bâtiment:")
display(stats_by_type.style.background_gradient(cmap='Greens', subset=['Empreinte (tCO2)']))

 Statistiques Moyennes par Type de Bâtiment:


,Superficie Moy. (m²),Occupants Moy.,Conso Élec. (kWh),Conso Gaz (kWh),Conso Eau (m³),Empreinte (tCO2)
type_batiment,,,,,,
Appartement,1257.200000,130.880000,118493.860000,99813.610000,6477.250000,30.270000
Bureau,1665.630000,113.620000,259997.690000,85699.210000,5739.300000,35.180000
Commerce,1587.150000,121.560000,316576.700000,95375.250000,5798.440000,40.980000
Entrepôt,1579.080000,143.980000,121465.200000,66667.180000,7124.310000,23.130000
Hôpital,1603.990000,107.540000,371820.770000,179604.510000,5281.220000,63.730000
Hôtel,1335.790000,144.640000,223041.180000,124621.380000,6978.300000,41.900000
Industrie,1397.820000,132.820000,354622.420000,185588.740000,6666.770000,63.990000
Maison Particulière,1434.940000,123.080000,142207.400000,130088.490000,6007.380000,39.140000
Résidence,1435.840000,123.140000,148721.780000,126750.640000,5903.940000,38.590000
